In [1]:
# required libraries
import os
import pickle
import torch
import torch.nn as nn

import numpy as np
import pandas as pd
import math

import data.processor as preprocess

import mlmodel.mlp as model_mlp
import mlmodel.softdt as model_softdt
import mlmodel.tab_transformer as model_tabtrans
import mlmodel.simple_vae as model_simple_vae

processed_data_path = os.path.join(os.getcwd(), 'data/preprocessed/')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

from attack.vae_deltaz import VAEDeltaZAttack
import attack.run_gridsearch as run_gridsearch

# Remove the Future warning from pytorch
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
LOAD_EXIST = True

In [3]:
def pick_true_prediction(model, data):
    X_test_tensor, y_test_tensor = data

    with torch.no_grad():
        model.eval()
        test_outputs = model(X_test_tensor.to(device))
        predicted = torch.argmax(test_outputs, dim=1).float()
        prediction = (predicted == y_test_tensor.to(device))

        # Create tensors to hold true predictions' indices, inputs, and targets
        indices_of_true_predictions = []
        X_of_true_predictions = []
        y_of_true_predictions = []

        for i in range(len(prediction)):
            if prediction[i]:
                indices_of_true_predictions.append(i)
                X_of_true_predictions.append(X_test_tensor[i])
                y_of_true_predictions.append(y_test_tensor[i])

        return (
            torch.tensor(indices_of_true_predictions),
            torch.stack(X_of_true_predictions),
            torch.stack(y_of_true_predictions),
        )


In [4]:
parameter_grid_baseline = {
    '_lambda': [1], # Regularization parameter, 0.01, 0.03, 0.1, 0.3, 0.5, 0.7, 1 # 0.6, 0.7, 0.8, 0.9, 1
    'lr': [0.1],  # Learning rates to try, 0.01, 0.03, 0.1, 0.15, 0.2, 0.25, 0.3
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'p_norm': [2], 
}

In [5]:
parameter_grid_search = {
    '_lambda': [0.6, 0.7, 0.8, 0.9, 1], # Regularization parameter,  0.6, 0.7, 0.8, 0.9, 1
    'lr': [0.01, 0.03, 0.1, 0.15, 0.2, 0.25, 0.3],  # Learning rates to try, 0.01, 0.03, 0.1, 0.15, 0.2, 0.25, 0.3
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'p_norm': [1, 2], 
}

### Adult

In [12]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/adult')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([21112, 12]), y_train: torch.Size([21112])
X_val: torch.Size([3017, 12]), y_val: torch.Size([3017])
X_test: torch.Size([6033, 12]), y_test: torch.Size([6033])
The num of embedding dim for Transformer is: 10
The num of categorical features is: 7
The num of continues features is: 5
The num of total features is: 12
The num of binary features is: 1
Combined embedding dims: [(16, 4), (7, 3), (7, 3), (14, 4), (6, 3), (5, 3), (41, 7)]


Load VAE model

In [13]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "adult",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-3,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/adult_0.001_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(16, 4)
    (1-2): 2 x Embedding(7, 3)
    (3): Embedding(14, 4)
    (4): Embedding(6, 3)
    (5): Embedding(5, 3)
    (6): Embedding(41, 7)
  )
  (encoder): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0

Load ml model

In [14]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "adult",
    "epochs": 150,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_adult.pt


In [15]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "adult",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=2, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_adult.pt


In [16]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "adult",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                      num_continuous=continues_num,
                                      dim=trans_dim,
                                      depth=3,
                                      heads=4,
                                      dim_head=16,
                                      dim_out=2,
                                      ff_dropout=0.2,
                                      attn_dropout=0.2,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-3, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_adult.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [20]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEDeltaZAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="adult",                         # Dataset name
        model=model_str,                               # Model name
        attack="deltaz",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 5151
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Successfully loaded existing results for parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.734,
        "mean_l2_distance": 1.4811991453170776,
        "mean_linf_distance": 0.5885276794433594,
        "perturbed_features_latent": 16.0,
        "perturbed_features_input": 5.2425068119891005,
        "mean_confidence_successful": 0.8886237685737227,
        "mean_confidence_unsuccessful": 0.18807054417473928,
        "min_confidence_unsuccessful": 0.0,
        "max_confidence_unsuccessful": 0.4990055561065674,
        "mean_md_d

Learning DeltaZ: 100%|██████████| 300/300 [01:11<00:00,  4.17it/s, loss=2.489765, reg=0.025673]


DeltaZ learned with norm: 0.015453


Generating DeltaZ Adversarial Examples: 100%|██████████| 500/500 [00:01<00:00, 254.37it/s]

Total successful adversarial examples: 19/500 (3.80%)
Adversarial examples confidence: Mean - 0.2064, Max - 0.8244, Min - 0.0001
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.038,
        "mean_l2_distance": 0.015452717430889606,
        "mean_linf_distance": 0.006249125115573406,
        "perturbed_features_latent": 16.0,
        "perturbed_features_input": 4.315789473684211,
        "mean_confidence_successful": 0.5859490640853581,
        "mean_confidence_unsuccessful": 0.19141664351346338,
        "min_confidence_unsuccessful": 6.973743438720703e-05,
        "max_confidence_unsuccessful": 0.4990459084510803,
        "mean_md_distance": 3.1409701424598886,
        "outlier_rate": 0.0,
        "outliers": "0/19"
    }
}

Best Parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Best Metrics: {'success_rate':

Run hyperparameter search

In [12]:
for model_str in ["MLP"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_search['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEDeltaZAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="adult",                         # Dataset name
        model=model_str,                               # Model name
        attack="deltaz",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_search,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 5151
------------------------------------------------------------
Running combination 1/70

Testing parameters: {'_lambda': 0.6, 'lr': 0.01, 'max_iter': 300, 'sample_num': 500, 'p_norm': 1}
Successfully loaded existing results for parameters: {'_lambda': 0.6, 'lr': 0.01, 'max_iter': 300, 'sample_num': 500, 'p_norm': 1}
Results: 
{
    "params": {
        "_lambda": 0.6,
        "lr": 0.01,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 1
    },
    "metrics": {
        "success_rate": 0.34,
        "mean_l2_distance": 1.794392704963684,
        "mean_linf_distance": 1.7943657636642456,
        "perturbed_features_latent": 16.0,
        "perturbed_features_input": 4.482352941176471,
        "mean_confidence_successful": 0.711245274004143,
        "mean_confidence_unsuccessful": 0.16217193585453613,
        "min_confidence_unsuccessful": 0.0,
        "max_confidence_unsuccessful": 0.49976760149002075,
        "me

### phishing_url

In [13]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/phishing_url')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([8001, 86]), y_train: torch.Size([8001])
X_val: torch.Size([1143, 86]), y_val: torch.Size([1143])
X_test: torch.Size([2286, 86]), y_test: torch.Size([2286])
The num of embedding dim for Transformer is: 2
The num of categorical features is: 1
The num of continues features is: 85
The num of total features is: 86
The num of binary features is: 28
Combined embedding dims: [(3, 2)]


In [14]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "phishing_url",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/phishing_url_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(3, 2)
  )
  (encoder): Sequential(
    (0): Linear(in_features=87, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=16, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_f

Load ml model

In [15]:

mlp_config = {
    "model": "MLP",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_phishing_url.pt


In [16]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_phishing_url.pt


In [17]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "phishing_url",
    "epochs": 50,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_phishing_url.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [18]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEDeltaZAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="phishing_url",                         # Dataset name
        model=model_str,                               # Model name
        attack="deltaz",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2198
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Successfully loaded existing results for parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.826,
        "mean_l2_distance": 1.1528176069259644,
        "mean_linf_distance": 0.7897940278053284,
        "perturbed_features_latent": 16.0,
        "perturbed_features_input": 59.92978208232446,
        "mean_confidence_successful": 0.9648830322629082,
        "mean_confidence_unsuccessful": 0.09572220122677157,
        "min_confidence_unsuccessful": 0.0,
        "max_confidence_unsuccessful": 0.4918518662452698,
        "mean_md_di

### PenDigit

In [19]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/pendigit')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([7693, 16]), y_train: torch.Size([7693])
X_val: torch.Size([1100, 16]), y_val: torch.Size([1100])
X_test: torch.Size([2199, 16]), y_test: torch.Size([2199])
The num of embedding dim for Transformer is: 0
The num of categorical features is: 0
The num of continues features is: 16
The num of total features is: 16
The num of binary features is: 0
Combined embedding dims: []


Load VAE model

In [20]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
    num_classes=10
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/pendigit_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList()
  (encoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=8, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=64, bias=True)
      (3): Re

Load ml model

In [21]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=10,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_pendigit.pt


In [22]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=10, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_pendigit.pt


In [23]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=[],
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=10,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')


Model loaded from models/TabTransformer_pendigit.pt


/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Apply attacks - MLP + SoftDT + Tab Transfomer

In [24]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEDeltaZAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="pendigit",                         # Dataset name
        model=model_str,                               # Model name
        attack="deltaz",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2154
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Could not load existing results: No files found in attack folder: adversarial_examples/pendigit/MLP/deltaz/Try_500_inputs/_lambda_1_p_2_lr_0.1_max_iter_300
Running attack instead for parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Folder adversarial_examples/pendigit/MLP/deltaz/Try_500_inputs/_lambda_1_p_2_lr_0.1_max_iter_300 already exists. Overwriting...
Learning deltaZ transformation vector...


Learning DeltaZ:   0%|          | 0/300 [00:00<?, ?it/s]


IndexError: Target -7 is out of bounds.

### Mushroom

In [25]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/mushroom')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([3950, 22]), y_train: torch.Size([3950])
X_val: torch.Size([565, 22]), y_val: torch.Size([565])
X_test: torch.Size([1129, 22]), y_test: torch.Size([1129])
The num of embedding dim for Transformer is: 10
The num of categorical features is: 18
The num of continues features is: 4
The num of total features is: 22
The num of binary features is: 4
Combined embedding dims: [(6, 3), (4, 2), (8, 3), (7, 3), (2, 2), (2, 2), (9, 3), (4, 2), (4, 2), (4, 2), (7, 3), (7, 3), (2, 2), (3, 2), (4, 2), (6, 3), (6, 3), (6, 3)]


In [26]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "mushroom",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 10,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/mushroom_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(6, 3)
    (1): Embedding(4, 2)
    (2): Embedding(8, 3)
    (3): Embedding(7, 3)
    (4-5): 2 x Embedding(2, 2)
    (6): Embedding(9, 3)
    (7-9): 3 x Embedding(4, 2)
    (10-11): 2 x Embedding(7, 3)
    (12): Embedding(2, 2)
    (13): Embedding(3, 2)
    (14): Embedding(4, 2)
    (15-17): 3 x Embedding(6, 3)
  )
  (encoder): Sequential(
    (0): Linear(in_features=49, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=10, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=10, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in

Load ml model

In [27]:

mlp_config = {
    "model": "MLP",
    "dataset": "mushroom",
    "epochs": 50,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_mushroom.pt


In [28]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "mushroom",
    "epochs": 75,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_mushroom.pt


In [29]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "mushroom",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=2,
                                         ff_dropout=0.2,
                                         attn_dropout=0.2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_mushroom.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [30]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEPGDAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="mushroom",                         # Dataset name
        model=model_str,                               # Model name
        attack="pgd",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 1129
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'epsilon': 0.5, 'alpha': 0.05, 'num_steps': 50, 'sample_num': 500}
Could not load existing results: Attack folder does not exist: adversarial_examples/mushroom/MLP/pgd/Try_500_inputs/_epsilon_0.5_alpha_0.05_steps_50
Running attack instead for parameters: {'epsilon': 0.5, 'alpha': 0.05, 'num_steps': 50, 'sample_num': 500}


Generating PGD Adversarial Examples:   0%|          | 0/500 [00:00<?, ?it/s]

Generating PGD Adversarial Examples: 100%|██████████| 500/500 [00:33<00:00, 14.82it/s]


Total successful adversarial examples: 8/500 (1.60%)
Adversarial examples confidence: Mean - 0.0168, Max - 1.0000, Min - 0.0000
Results: 
{
    "params": {
        "epsilon": 0.5,
        "alpha": 0.05,
        "num_steps": 50,
        "sample_num": 500
    },
    "metrics": {
        "success_rate": 0.016,
        "mean_l2_distance": 1.0126368999481201,
        "mean_linf_distance": 0.4759799838066101,
        "perturbed_features_latent": 10.0,
        "perturbed_features_input": 9.0,
        "mean_confidence_successful": 0.9999510653474452,
        "mean_confidence_unsuccessful": 0.0007973361790664797,
        "min_confidence_unsuccessful": 0.0,
        "max_confidence_unsuccessful": 0.36830389499664307,
        "mean_md_distance": 4.500467443522547,
        "outlier_rate": 0.625,
        "outliers": "5/8"
    }
}

Best Parameters: {'epsilon': 0.5, 'alpha': 0.05, 'num_steps': 50, 'sample_num': 500}
Best Metrics: {'success_rate': 0.016, 'mean_l2_distance': 1.0126369, 'mean_linf_distan

Generating PGD Adversarial Examples: 100%|██████████| 500/500 [01:45<00:00,  4.73it/s]


Total successful adversarial examples: 7/500 (1.40%)
Adversarial examples confidence: Mean - 0.1777, Max - 0.8723, Min - 0.1260
Results: 
{
    "params": {
        "epsilon": 0.5,
        "alpha": 0.05,
        "num_steps": 50,
        "sample_num": 500
    },
    "metrics": {
        "success_rate": 0.014,
        "mean_l2_distance": 0.9963093400001526,
        "mean_linf_distance": 0.47963428497314453,
        "perturbed_features_latent": 10.0,
        "perturbed_features_input": 9.428571428571429,
        "mean_confidence_successful": 0.8429082610777446,
        "mean_confidence_unsuccessful": 0.16824007433278082,
        "min_confidence_unsuccessful": 0.12597113847732544,
        "max_confidence_unsuccessful": 0.4307435154914856,
        "mean_md_distance": 5.100754793563659,
        "outlier_rate": 0.5714285714285714,
        "outliers": "4/7"
    }
}

Best Parameters: {'epsilon': 0.5, 'alpha': 0.05, 'num_steps': 50, 'sample_num': 500}
Best Metrics: {'success_rate': 0.014, 'mean_l

Generating PGD Adversarial Examples: 100%|██████████| 500/500 [03:05<00:00,  2.69it/s]

Total successful adversarial examples: 6/500 (1.20%)
Adversarial examples confidence: Mean - 0.0190, Max - 0.9995, Min - 0.0001
Results: 
{
    "params": {
        "epsilon": 0.5,
        "alpha": 0.05,
        "num_steps": 50,
        "sample_num": 500
    },
    "metrics": {
        "success_rate": 0.012,
        "mean_l2_distance": 0.9354608654975891,
        "mean_linf_distance": 0.4710053503513336,
        "perturbed_features_latent": 10.0,
        "perturbed_features_input": 8.166666666666666,
        "mean_confidence_successful": 0.9862512640005056,
        "mean_confidence_unsuccessful": 0.007270925078797437,
        "min_confidence_unsuccessful": 5.1021575927734375e-05,
        "max_confidence_unsuccessful": 0.2499677538871765,
        "mean_md_distance": 3.7162699342411325,
        "outlier_rate": 0.16666666666666666,
        "outliers": "1/6"
    }
}

Best Parameters: {'epsilon': 0.5, 'alpha': 0.05, 'num_steps': 50, 'sample_num': 500}
Best Metrics: {'success_rate': 0.012, 'm

### electricity

In [6]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/electricity')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([31717, 8]), y_train: torch.Size([31717])
X_val: torch.Size([4532, 8]), y_val: torch.Size([4532])
X_test: torch.Size([9063, 8]), y_test: torch.Size([9063])
The num of embedding dim for Transformer is: 3
The num of categorical features is: 1
The num of continues features is: 7
The num of total features is: 8
The num of binary features is: 0
Combined embedding dims: [(7, 3)]


In [7]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "electricity",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [16],
    "kl_weight": 1e-4,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/electricity_0.0001_100_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(7, 3)
  )
  (encoder): Sequential(
    (0): Linear(in_features=10, out_features=16, bias=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=16, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=16, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=8, out_features=16, bias=True)
      (1): ReLU()
    )
    (1): Linear(in_features=16, out_features=8, bias=True)
    (2): Sigmoid()
  )
  (cat_decoder_layers): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=16, out_features=16, bias=True)
      (1): ReLU()
      (2): Linear(in_features=16, out_features=7, bias=True)
    )
  )
  (num_decoder): Sequential(
  

Load ml model

In [8]:

mlp_config = {
    "model": "MLP",
    "dataset": "electricity",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.1
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_electricity.pt


In [9]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "electricity",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-2, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_electricity.pt


In [10]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "electricity",
    "epochs": 100,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=2,
                                         heads=3,
                                         dim_head=4,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-3, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_electricity.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [ ]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEDeltaZAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="electricity",                         # Dataset name
        model=model_str,                               # Model name
        attack="deltaz",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 7351
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Could not load existing results: Attack folder does not exist: adversarial_examples/electricity/MLP/deltaz/Try_500_inputs/_lambda_1_p_2_lr_0.1_max_iter_300
Running attack instead for parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Learning deltaZ transformation vector...


Learning DeltaZ: 100%|██████████| 300/300 [00:02<00:00, 124.33it/s, loss=1.581367, reg=0.813809]


DeltaZ learned with norm: 0.803456


Generating DeltaZ Adversarial Examples: 100%|██████████| 500/500 [00:00<00:00, 1258.06it/s]


Total successful adversarial examples: 329/500 (65.80%)
Adversarial examples confidence: Mean - 0.5924, Max - 0.9934, Min - 0.0000
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.658,
        "mean_l2_distance": 0.8034561276435852,
        "mean_linf_distance": 0.5764158368110657,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 7.0,
        "mean_confidence_successful": 0.7672753001755743,
        "mean_confidence_unsuccessful": 0.25606144311135276,
        "min_confidence_unsuccessful": 1.9073486328125e-06,
        "max_confidence_unsuccessful": 0.4984099864959717,
        "mean_md_distance": 2.446728905791853,
        "outlier_rate": 0.0182370820668693,
        "outliers": "6/329"
    }
}

Best Parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Best Metrics: {'success_rate': 0.6

...

Total successful adversarial examples: 331/500 (66.20%)
Adversarial examples confidence: Mean - 0.5864, Max - 0.9843, Min - 0.0023
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.662,
        "mean_l2_distance": 0.8441936373710632,
        "mean_linf_distance": 0.6367397904396057,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 7.0,
        "mean_confidence_successful": 0.7599743870494949,
        "mean_confidence_unsuccessful": 0.2463067378518144,
        "min_confidence_unsuccessful": 0.002333521842956543,
        "max_confidence_unsuccessful": 0.4962881803512573,
        "mean_md_distance": 2.453677625489274,
        "outlier_rate": 0.021148036253776436,
        "outliers": "7/331"
    }
}

Best Parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Best Metrics: {'success_rate': 0

Learning DeltaZ: 100%|██████████| 300/300 [00:10<00:00, 28.26it/s, loss=1.693905, reg=0.922511]


DeltaZ learned with norm: 0.920135


Generating DeltaZ Adversarial Examples: 100%|██████████| 500/500 [00:01<00:00, 408.77it/s]

Total successful adversarial examples: 311/500 (62.20%)
Adversarial examples confidence: Mean - 0.5838, Max - 0.9978, Min - 0.0057
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.622,
        "mean_l2_distance": 0.920135498046875,
        "mean_linf_distance": 0.6780260801315308,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 7.0,
        "mean_confidence_successful": 0.7786095903690761,
        "mean_confidence_unsuccessful": 0.26335364800912364,
        "min_confidence_unsuccessful": 0.005714237689971924,
        "max_confidence_unsuccessful": 0.499613881111145,
        "mean_md_distance": 2.4371098761467103,
        "outlier_rate": 0.00964630225080386,
        "outliers": "3/311"
    }
}

Best Parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Best Metrics: {'success_rate': 0.

### german_credit

In [6]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/german_credit')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([700, 19]), y_train: torch.Size([700])
X_val: torch.Size([100, 19]), y_val: torch.Size([100])
X_test: torch.Size([200, 19]), y_test: torch.Size([200])
The num of embedding dim for Transformer is: 8
The num of categorical features is: 11
The num of continues features is: 8
The num of total features is: 19
The num of binary features is: 2
Combined embedding dims: [(4, 2), (5, 3), (10, 4), (5, 3), (5, 3), (4, 2), (3, 2), (4, 2), (3, 2), (3, 2), (4, 2)]


In [7]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "german_credit",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/german_credit_0.01_100_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(4, 2)
    (1): Embedding(5, 3)
    (2): Embedding(10, 4)
    (3-4): 2 x Embedding(5, 3)
    (5): Embedding(4, 2)
    (6): Embedding(3, 2)
    (7): Embedding(4, 2)
    (8-9): 2 x Embedding(3, 2)
    (10): Embedding(4, 2)
  )
  (encoder): Sequential(
    (0): Linear(in_features=35, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_f

Load ml model

In [8]:

mlp_config = {
    "model": "MLP",
    "dataset": "german_credit",
    "epochs": 50,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.1
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_german_credit.pt


In [9]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "german_credit",
    "epochs": 30,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-2, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_german_credit.pt


In [10]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "german_credit",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=2,
                                         heads=3,
                                         dim_head=5,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-2, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_german_credit.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [11]:

for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEDeltaZAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="german_credit",                         # Dataset name
        model=model_str,                               # Model name
        attack="deltaz",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
        override_sample_num=len(sampled_prediction_y_tensor)
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 149
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Could not load existing results: Attack folder does not exist: adversarial_examples/german_credit/MLP/deltaz/Try_149_inputs/_lambda_1_p_2_lr_0.1_max_iter_300
Running attack instead for parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Learning deltaZ transformation vector...


Learning DeltaZ: 100%|██████████| 300/300 [00:02<00:00, 109.13it/s, loss=1.664260, reg=0.009648]


DeltaZ learned with norm: 0.012151


Generating DeltaZ Adversarial Examples: 100%|██████████| 149/149 [00:00<00:00, 835.59it/s]


Total successful adversarial examples: 12/149 (8.05%)
Adversarial examples confidence: Mean - 0.2345, Max - 0.6135, Min - 0.0257
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.08053691275167785,
        "mean_l2_distance": 0.012151292525231838,
        "mean_linf_distance": 0.006569643970578909,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 9.833333333333334,
        "mean_confidence_successful": 0.5460730617245039,
        "mean_confidence_unsuccessful": 0.20719753782244493,
        "min_confidence_unsuccessful": 0.025654733180999756,
        "max_confidence_unsuccessful": 0.4891544580459595,
        "mean_md_distance": 2.45401708464864,
        "outlier_rate": 0.0,
        "outliers": "0/12"
    }
}

Best Parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Best Metrics: {'succ

...

Total successful adversarial examples: 10/153 (6.54%)
Adversarial examples confidence: Mean - 0.2838, Max - 0.6206, Min - 0.1625
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.06535947712418301,
        "mean_l2_distance": 0.00822609756141901,
        "mean_linf_distance": 0.004830536432564259,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 9.6,
        "mean_confidence_successful": 0.5792180597782135,
        "mean_confidence_unsuccessful": 0.26315126510766834,
        "min_confidence_unsuccessful": 0.16250872611999512,
        "max_confidence_unsuccessful": 0.499064564704895,
        "mean_md_distance": 2.2745112054423036,
        "outlier_rate": 0.0,
        "outliers": "0/10"
    }
}

Best Parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Best Metrics: {'success_rate': 0.06

Learning DeltaZ: 100%|██████████| 300/300 [00:16<00:00, 17.85it/s, loss=2.386602, reg=0.018223]


DeltaZ learned with norm: 0.015610


Generating DeltaZ Adversarial Examples: 100%|██████████| 149/149 [00:00<00:00, 364.04it/s]

Total successful adversarial examples: 16/149 (10.74%)
Adversarial examples confidence: Mean - 0.2089, Max - 0.7749, Min - 0.0036
Results: 
{
    "params": {
        "_lambda": 1,
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "p_norm": 2
    },
    "metrics": {
        "success_rate": 0.10738255033557047,
        "mean_l2_distance": 0.015610392205417156,
        "mean_linf_distance": 0.012311376631259918,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 10.0625,
        "mean_confidence_successful": 0.6338622597977519,
        "mean_confidence_unsuccessful": 0.15776088587323525,
        "min_confidence_unsuccessful": 0.003597736358642578,
        "max_confidence_unsuccessful": 0.48089122772216797,
        "mean_md_distance": 2.6121726528980016,
        "outlier_rate": 0.0625,
        "outliers": "1/16"
    }
}

Best Parameters: {'_lambda': 1, 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'p_norm': 2}
Best Metrics: {'success